# House Price Model Final (Model Only)

This notebook is model-only.
Feature engineering and location encoding are handled in [notebooks/arenda-az-Data-Preprocessing.ipynb](notebooks/arenda-az-Data-Preprocessing.ipynb), which exports model inputs.

Workflow:
1. Load model-ready base feature data.
2. Load pre-encoded location features (OOF export).
3. Train RF baseline on combined features.
4. Train OOF-selected ensemble (ExtraTrees + LightGBM).
5. Run 10-seed stability and save final score artifacts.

In [51]:
import numpy as np
import pandas as pd
from scipy.stats import randint

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42

In [52]:
df = pd.read_csv('../data/satilir_model_base_v3.csv')
print('Shape:', df.shape)
print('Columns:', len(df.columns))
print('Missing values:', int(df.isna().sum().sum()))
df.head()

Shape: (4416, 51)
Columns: 51
Missing values: 0


,yes_no_binary__has_document,yes_no_binary__avtodayanacaq,yes_no_binary__balkon,yes_no_binary__duzelme,yes_no_binary__esyali,yes_no_binary__hovuz,yes_no_binary__internet,yes_no_binary__isiq,yes_no_binary__kabel_tv,yes_no_binary__kombi,...,num__land_to_area_ratio_v3,num__land_x_area_v3,num__area_bin_x_rooms_bin_v3,num__amenity_count_v3,num__comfort_score_v3,num__utility_score_v3,num__comfort_x_rooms_v3,num__utility_per_area_v3,num__log_raion_freq_v3,num__raionfreq_x_logarea_v3
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,12.0,12.0,7.0,5.0,21.0,0.076923,0.073331,0.318778
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.073331,0.304906
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,33.0,2.0,0.0,0.0,0.0,0.000000,0.064241,0.306212
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.000000,0.128594,0.522382
4,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,22.0,10.0,5.0,5.0,15.0,0.072464,0.128594,0.583014


In [53]:
def winsorize_series_v3(s: pd.Series, low_q: float = 0.01, high_q: float = 0.99) -> pd.Series:
    low_v, high_v = s.quantile([low_q, high_q])
    return s.clip(low_v, high_v)

## Step 1: Load Pre-Encoded Location Features

Location features are prepared in [notebooks/arenda-az-Data-Preprocessing.ipynb](notebooks/arenda-az-Data-Preprocessing.ipynb) and exported to ../data/satilir_location_encoded_oof_v6.csv.

In [54]:
# Load leakage-safer OOF location features generated in preprocessing notebook
loc_encoded_path = '../data/satilir_location_encoded_oof_v6.csv'
loc_encoded_all = pd.read_csv(loc_encoded_path)

print('Encoded location file:', loc_encoded_path)
print('Encoded location rows:', len(loc_encoded_all), '| Model rows:', len(df))
if len(loc_encoded_all) != len(df):
    raise ValueError(f'Row mismatch: encoded={len(loc_encoded_all)}, model={len(df)}')
print('Encoded location columns:', len(loc_encoded_all.columns))
print('Total missing values in encoded location:', int(loc_encoded_all.isna().sum().sum()))
loc_encoded_all.head()

Encoded location file: ../data/satilir_location_encoded_oof_v6.csv
Encoded location rows: 4416 | Model rows: 4416
Encoded location columns: 10
Total missing values in encoded location: 0


,num__city_te_v6,num__district_te_v6,num__neighborhood_te_v6,num__metro_te_v6,num__metro_present_v6,num__location_te_mean_v6,num__district_to_city_ratio_v6,num__metro_to_district_ratio_v6,num__locmean_x_logarea_v6,num__locmean_x_rooms_v6
0,171730.550344,238952.016561,179330.220742,258853.230543,1.0,212216.504547,1.391436,1.083285,889113.884614,636649.513642
1,172053.254339,245507.264770,179896.797849,235281.628484,1.0,208184.736360,1.426926,0.958349,834265.602676,416369.472721
2,173673.430802,170711.235372,144781.721788,112477.509204,0.0,150410.974292,0.982944,0.658876,694164.773411,601643.897166
3,107130.470176,105749.561378,179545.887369,112477.509204,0.0,126225.857032,0.987110,1.063622,480499.235201,126225.857032
4,108615.386787,106696.623419,179429.629097,112868.037740,0.0,126902.419261,0.982334,1.057841,539144.324434,380707.257782


In [55]:
# Train with pre-encoded location features exported from preprocessing notebook
X_base_loc = df.drop(columns=['num__price']).copy()
y_loc = df['num__price'].copy()

X_train_loc_base, X_test_loc_base, y_train_loc, y_test_loc = train_test_split(
    X_base_loc,
    y_loc,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

y_train_loc_w = winsorize_series_v3(y_train_loc, 0.01, 0.99)

loc_train_feat = loc_encoded_all.loc[X_train_loc_base.index]
loc_test_feat = loc_encoded_all.loc[X_test_loc_base.index]

X_train_loc = pd.concat([X_train_loc_base, loc_train_feat], axis=1)
X_test_loc = pd.concat([X_test_loc_base, loc_test_feat], axis=1)

loc_param_dist = {
    'n_estimators': randint(1400, 3000),
    'max_depth': [None, 16, 20, 24, 32],
    'max_features': [0.6, 0.7, 0.8, 0.9],
    'min_samples_split': randint(3, 9),
    'min_samples_leaf': randint(1, 4),
    'bootstrap': [True],
}

rf_loc_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=loc_param_dist,
    n_iter=35,
    scoring='r2',
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

rf_loc_search.fit(X_train_loc, y_train_loc_w)
best_rf_loc = rf_loc_search.best_estimator_
best_rf_loc_params = rf_loc_search.best_params_

pred_loc = np.clip(best_rf_loc.predict(X_test_loc), 0, None)

print('--- RF + Pre-Encoded Location Signal ---')
print('Best CV R2 :', round(rf_loc_search.best_score_, 4))
print('Test R2    :', round(r2_score(y_test_loc, pred_loc), 4))
print(f"Test RMSE  : ${np.sqrt(mean_squared_error(y_test_loc, pred_loc)):,.0f}")
print(f"Test MAE   : ${mean_absolute_error(y_test_loc, pred_loc):,.0f}")
print('Best Params:', best_rf_loc_params)
print('Feature count with pre-encoded location features:', X_train_loc.shape[1])

Fitting 5 folds for each of 35 candidates, totalling 175 fits
--- RF + Pre-Encoded Location Signal ---
Best CV R2 : 0.7654
Test R2    : 0.8012
Test RMSE  : $40,526
Test MAE   : $27,805
Best Params: {'bootstrap': True, 'max_depth': 32, 'max_features': 0.6, 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 2767}
Feature count with pre-encoded location features: 60


## Step 2: Final Ensemble Improvement

Train RF baseline with location features, then train an OOF-selected two-model ensemble and validate with 10-seed stability.

In [62]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor

# OOF ensemble on the same train/test split created in the location-enhanced RF cell
# Uses training-only folds for weight selection (no test leakage in weight search).
# Raw target values are used here to preserve full price signal.
cv_blend = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

base_models = {
    'rf': RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **best_rf_loc_params,
    ),
    'et': ExtraTreesRegressor(
        n_estimators=2200,
        max_depth=None,
        max_features=0.7,
        min_samples_split=2,
        min_samples_leaf=1,
        bootstrap=False,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'lgbm': LGBMRegressor(
        n_estimators=1600,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.85,
        colsample_bytree=0.8,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),
}

oof_pred = {k: np.zeros(len(X_train_loc), dtype=float) for k in base_models}
for tr_idx, va_idx in cv_blend.split(X_train_loc):
    X_tr_f = X_train_loc.iloc[tr_idx]
    X_va_f = X_train_loc.iloc[va_idx]
    y_tr_f = y_train_loc.iloc[tr_idx]

    for name, model in base_models.items():
        m = clone(model)
        m.fit(X_tr_f, y_tr_f)
        oof_pred[name][va_idx] = np.clip(m.predict(X_va_f), 0, None)

blend_grid = np.linspace(0.0, 1.0, 21)
best_combo = ('rf', 1.0, 'et', 0.0)
best_oof_r2 = -1e9
for left, right in [('rf', 'et'), ('rf', 'lgbm'), ('et', 'lgbm')]:
    for w in blend_grid:
        pred_bl = w * oof_pred[left] + (1.0 - w) * oof_pred[right]
        score = r2_score(y_train_loc, pred_bl)
        if score > best_oof_r2:
            best_oof_r2 = score
            best_combo = (left, float(w), right, float(1.0 - w))

fitted_full = {}
for name, model in base_models.items():
    m = clone(model)
    m.fit(X_train_loc, y_train_loc)
    fitted_full[name] = m

left, w_left, right, w_right = best_combo
pred_ens = np.clip(
    w_left * fitted_full[left].predict(X_test_loc)
    + w_right * fitted_full[right].predict(X_test_loc),
    0,
    None,
)

rf_base_r2 = r2_score(y_test_loc, pred_loc)
ens_r2 = r2_score(y_test_loc, pred_ens)
ens_rmse = np.sqrt(mean_squared_error(y_test_loc, pred_ens))
ens_mae = mean_absolute_error(y_test_loc, pred_ens)

print('--- OOF Ensemble on Location-Enhanced Features ---')
print(f'Best OOF pair : {left} ({w_left:.2f}) + {right} ({w_right:.2f})')
print(f'Best OOF R2   : {best_oof_r2:.4f}')
print(f'RF Test R2    : {rf_base_r2:.4f}')
print(f'Ensemble R2   : {ens_r2:.4f}')
print(f'Delta R2      : {ens_r2 - rf_base_r2:+.4f}')
print(f'Ensemble RMSE : ${ens_rmse:,.0f}')
print(f'Ensemble MAE  : ${ens_mae:,.0f}')

--- OOF Ensemble on Location-Enhanced Features ---
Best OOF pair : et (0.60) + lgbm (0.40)
Best OOF R2   : 0.7762
RF Test R2    : 0.8012
Ensemble R2   : 0.8243
Delta R2      : +0.0231
Ensemble RMSE : $38,100
Ensemble MAE  : $26,236


In [63]:
# 10-seed stability check for the selected OOF ensemble pair
ens_seed_rows = []

for s in range(1, 11):
    X_tr_b, X_te_b, y_tr, y_te = train_test_split(
        X_base_loc,
        y_loc,
        test_size=0.2,
        random_state=s,
    )

    X_tr = pd.concat([X_tr_b, loc_encoded_all.loc[X_tr_b.index]], axis=1)
    X_te = pd.concat([X_te_b, loc_encoded_all.loc[X_te_b.index]], axis=1)

    left_model = clone(base_models[left])
    right_model = clone(base_models[right])

    if hasattr(left_model, 'random_state'):
        left_model.set_params(random_state=s)
    if hasattr(right_model, 'random_state'):
        right_model.set_params(random_state=s)

    left_model.fit(X_tr, y_tr)
    right_model.fit(X_tr, y_tr)

    pred_ens_s = np.clip(
        w_left * left_model.predict(X_te) + w_right * right_model.predict(X_te),
        0,
        None,
    )

    ens_seed_rows.append({
        'seed': s,
        'test_r2': r2_score(y_te, pred_ens_s),
        'rmse': np.sqrt(mean_squared_error(y_te, pred_ens_s)),
        'mae': mean_absolute_error(y_te, pred_ens_s),
    })

ens_stability_df = pd.DataFrame(ens_seed_rows).sort_values('test_r2', ascending=False).reset_index(drop=True)
display(ens_stability_df)

print('\nOOF ensemble stability summary (10 seeds):')
print(f"Mean Test R2  : {ens_stability_df['test_r2'].mean():.4f}")
print(f"Median Test R2: {ens_stability_df['test_r2'].median():.4f}")
print(f"Best Test R2  : {ens_stability_df['test_r2'].max():.4f}")
print(f"Worst Test R2 : {ens_stability_df['test_r2'].min():.4f}")

,seed,test_r2,rmse,mae
0,1,0.809505,39468.848190,26231.337697
1,5,0.805386,41380.698038,26612.303435
2,10,0.804291,40405.723856,25651.621519
3,6,0.790424,42318.045444,27525.078554
4,9,0.784600,42187.063357,27089.445360
5,4,0.784036,40703.733998,26579.294347
6,2,0.779345,42638.813001,27300.583624
7,7,0.775443,40765.306131,26175.426204
8,8,0.774204,43437.513717,28108.607896
9,3,0.763832,43684.262913,28501.165630



OOF ensemble stability summary (10 seeds):
Mean Test R2  : 0.7871
Median Test R2: 0.7843
Best Test R2  : 0.8095
Worst Test R2 : 0.7638


In [64]:
# Step 3: Save final scores and metadata
from pathlib import Path
from datetime import datetime, timezone
import json

scores_dir = Path('../models/metrics')
scores_dir.mkdir(parents=True, exist_ok=True)

score_row = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
    'model_name': 'oof_ensemble_location_v2_raw_target',
    'rf_test_r2': float(rf_base_r2) if 'rf_base_r2' in globals() else float('nan'),
    'ensemble_test_r2': float(ens_r2) if 'ens_r2' in globals() else float('nan'),
    'ensemble_rmse': float(ens_rmse) if 'ens_rmse' in globals() else float('nan'),
    'ensemble_mae': float(ens_mae) if 'ens_mae' in globals() else float('nan'),
    'stability_mean_r2': float(ens_stability_df['test_r2'].mean()) if 'ens_stability_df' in globals() else float('nan'),
    'stability_median_r2': float(ens_stability_df['test_r2'].median()) if 'ens_stability_df' in globals() else float('nan'),
    'stability_best_r2': float(ens_stability_df['test_r2'].max()) if 'ens_stability_df' in globals() else float('nan'),
    'stability_worst_r2': float(ens_stability_df['test_r2'].min()) if 'ens_stability_df' in globals() else float('nan'),
    'best_pair_left': left if 'left' in globals() else None,
    'best_pair_weight_left': float(w_left) if 'w_left' in globals() else None,
    'best_pair_right': right if 'right' in globals() else None,
    'best_pair_weight_right': float(w_right) if 'w_right' in globals() else None,
}

latest_path = scores_dir / 'latest_scores.json'
with latest_path.open('w', encoding='utf-8') as f:
    json.dump(score_row, f, indent=2, ensure_ascii=False)

history_path = scores_dir / 'score_history.csv'
history_df = pd.DataFrame([score_row])
if history_path.exists():
    history_df.to_csv(history_path, mode='a', header=False, index=False)
else:
    history_df.to_csv(history_path, index=False)

print('Saved latest score to:', latest_path)
print('Appended score history to:', history_path)
print('Ensemble Test R2:', f"{score_row['ensemble_test_r2']:.4f}")
print('Stability Mean R2:', f"{score_row['stability_mean_r2']:.4f}")

Saved latest score to: ..\models\metrics\latest_scores.json
Appended score history to: ..\models\metrics\score_history.csv
Ensemble Test R2: 0.8243
Stability Mean R2: 0.7871
